## 35. 프로젝트 루트 설정

In [23]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)
print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\ai-data-analysis
데이터 폴더: c:\dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [24]:
import pandas as pd

# 데이터프레임 정의
customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")

## 37. 기본 구조와 주요 키 확인

In [25]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (301, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (766, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [26]:
# 병합 실습 전에 기준 테이블의 주요 키가 고유한지 확인합니다.
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():
    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## 38. Series와 DataFrame 선택

In [27]:
city_series = customers["city"]

customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]

print(type(city_series))
print(type(customer_view))

display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


## 39. 단일 조건 필터링

In [28]:
# 하나만 필터링해서 보겠다.
customers_over_30 = customers[
    customers["age"] >= 30
]

print(len(customers), len(customers_over_30))

display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,장춘자,F,32,대구,2023-11-12
2,3,김상현,F,61,성남,2025-02-12
3,4,김재호,F,55,울산,2025-03-28
5,6,한순자,F,32,성남,2024-01-26
6,7,이미숙,F,53,인천,2023-11-23


## 40. 복합 조건 필터링

In [29]:
# 30세 이상이면서 서울 거주:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울") # & 이므로 동시에 조건 만족해야함 
]

display(seoul_over_30.head()) # 5개 뽑아서 보여줌

# 서울 또는 부산:
seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"]) # isin() city열에서 서울 혹은 부산이 들어간 것
]

display(
    seoul_or_busan["city"].value_counts() # value_count() 중요한 메소드!!! 값의 개수를 count
)

# 완료 주문이 아닌 주문:
not_completed = orders[
    ~(orders["order_status"] == "completed") # ~ 는 여집합
]

display(
    not_completed["order_status"].value_counts(
        dropna=False
    )
)

,customer_id,name,gender,age,city,signup_date
8,9,우영미,M,69,서울,2025-04-05
14,15,서민준,M,69,서울,2024-09-27
29,30,윤우진,F,32,서울,2024-01-23
47,48,박승현,F,47,서울,2024-06-09
65,66,양영미,F,39,서울,2024-01-28


city
부산    16
서울    15
Name: count, dtype: int64

order_status
cancelled    65
refunded     52
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [30]:
# 필터링 후엔 꼭 정렬해서 봐야 함. order by

expensive_products = (
    products
    .sort_values("price", ascending=False) # sort_values() # ascending(오름차순)=False이므로 내림차순()
    .head(10) # 10개를 볼것
)

display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


## 42. 작업용 복사본과 파생 컬럼

In [31]:
order_items_work = order_items.copy() # 카피본 생성, 원본 보존을 위해.

# line_total 열 추가
# 데이터 프레임에 새로운 컬럼을 추가할 때는 기존 데이터 프레임을 수정하지 않는다.
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

# 결과 확인(간단 빠름):
print(order_items_work.head())
# 결과 확인(예쁜 버전):
display(
    order_items_work[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ]
    ].head()
)

   order_item_id  order_id  product_id  quantity  unit_price  line_total
0              1         1         100         3      102000      306000
1              2         1          87         5       25000      125000
2              3         1           7         3      142000      426000
3              4         1           9         3      193000      579000
4              5         2          72         4      189000      756000


,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


## 43. 수작업 검증

In [32]:
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]

print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True


## 44. 전체 주문상세 금액

In [33]:
all_order_amount = order_items_work["line_total"].sum()

print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255930000


현재 합계에는 취소 또는 환불 주문이 포함될 수 있으므로

완료 주문 기준 매출이 아니라 전체 주문상세 금액으로 표현한다.